
# 1. Load the Qwen model

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-7B-Instruct" # Hoặc phiên bản Qwen bạn đang dùng

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. Load Model tối ưu cho Kaggle T4 x 2
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,        # Dùng float16 rất hợp với GPU T4
    device_map="auto",                # Tự động phân bổ đều lên cả 2 GPU T4
    attn_implementation="sdpa"        # KHÔNG cần flash-attn, dùng SDPA thay thế
)

# Thử nghiệm chat
messages = [
    {"role": "user", "content": "Xin chào, bạn là ai?"}
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

generated_ids = model.generate(**model_inputs, max_new_tokens=512)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Xin chào! Tôi là Qwen, một mô hình ngôn ngữ lớn được tạo ra bởi Alibaba Cloud. Tôi có thể giúp bạn với nhiều câu hỏi và nhiệm vụ liên quan đến ngôn ngữ tự nhiên. Bạn cần hỗ trợ gì hôm nay không?


# 2. Read the dataset

In [5]:
import os

md_files = []

root_dir = "/kaggle/input/datasets/giang792/mini-exam-dataset"

for root, dirs, files in os.walk(root_dir):

    for file in files:

        if file.endswith(".md"):

            md_files.append(
                os.path.join(root, file)
            )

print(len(md_files))

6


# 3. Run the model

In [8]:
import json
import re

SYSTEM_PROMPT = """
You are an expert dataset builder specializing in data extraction.

Extract question-answer pairs from markdown.

Return ONLY valid JSON array. Do not wrap the response in markdown code blocks. 

JSON Format Template:
[
  {
    "id": "QA_001",
    "source_origin": "Heading or Section",
    "topic": "Subject",
    "difficulty": ["Easy"],
    "question": "Question text",
    "answer": "Answer text"
  }
]

No explanation.
"""

def extract_qa(md_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": md_text[:4000]} # Giới hạn 4000 ký tự đầu
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=1500, # Tăng một chút phòng trường hợp văn bản dài
        temperature=0.1,
        do_sample=False     # Đảm bảo tính nhất quán cao nhất cho JSON
    )

    # Lấy chính xác phần text được sinh ra mới
    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # --- BƯỚC LÀM SẠCH VÀ SỬA LỖI JSON (QUAN TRỌNG NHẤT) ---
    # 1. Xóa các cụm ```json hoặc ``` ở đầu/cuối nếu có
    response = re.sub(r'^```json\s*', '', response, flags=re.IGNORECASE)
    response = re.sub(r'^```\s*', '', response, flags=re.IGNORECASE)
    response = re.sub(r'\s*```$', '', response, flags=re.IGNORECASE)
    response = response.strip()

    # 2. Thử load JSON trực tiếp
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        # Nếu vẫn lỗi, cố gắng tìm khối mảng [ ... ] bên trong chuỗi văn bản bẩn
        match = re.search(r'\[\s*\{.*\}\s*\]', response, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        else:
            # In ra chuỗi lỗi để bạn dễ debug xem Qwen đang trả về cái gì
            print(f"\n[DEBUG] Không thể parse JSON từ phản hồi: \n{response}\n")
            raise

all_pairs = []

print(f"Bắt đầu xử lý {len(md_files)} files...")

for i, path in enumerate(md_files):
    with open(path, encoding="utf-8") as f:
        md = f.read()

    try:
        pairs = extract_qa(md)
        if isinstance(pairs, list): # Đảm bảo đầu ra là một danh sách
            all_pairs.extend(pairs)
    except Exception as e:
        # In rõ lỗi chi tiết thay vì chỉ hiện tên file ẩn giấu lỗi đi
        print(f"Lỗi tại file {path}: {str(e)}")

    if (i + 1) % 1 == 0: # Hoặc i % 100 tùy bạn, hiện tại có 6 file thì in từng file
        print(f"Đã xử lý xong file thứ {i+1}/{len(md_files)}")

print(f"\nHoàn thành! Tổng số cặp QA thu được: {len(all_pairs)}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Bắt đầu xử lý 6 files...
Đã xử lý xong file thứ 1/6
Đã xử lý xong file thứ 2/6
Đã xử lý xong file thứ 3/6
Đã xử lý xong file thứ 4/6
Đã xử lý xong file thứ 5/6
Đã xử lý xong file thứ 6/6

Hoàn thành! Tổng số cặp QA thu được: 18


In [21]:
import json

# Tên file bạn muốn lưu (ví dụ: output_dataset.json)
output_file_path = "dataset_qa_qwen.json"

# Mở file và ghi dữ liệu vào
with open(output_file_path, "w", encoding="utf-8") as f:
    json.dump(all_pairs, f, ensure_ascii=False, indent=4)

print(f"Đã lưu thành công {len(all_pairs)} cặp câu hỏi - trả lời vào file: {output_file_path}")

Đã lưu thành công 18 cặp câu hỏi - trả lời vào file: dataset_qa_qwen.json
